# 1 - Data Cleaning/First Preprocessing 

This project is going to start first getting a better understanding of the data and exploring it.

First making sure there is a common structure, and later visualize, explore and connect the data between them

## 1.1 Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

## 1.2 Charging The Data and Basic Preprocessing

On this dataset we have more than one .csv, so we are going to load them as we explore them, and one by one decide how to or if joining them into a bigger table

In [ ]:
folder = Path().resolve().parent
defecto = folder / "data" / "home-credit-default-risk"

### 1.2.1 application_train.csv

We are going to only use this one, as the test version does not have the target and we cant measure if it performs well on that partition.

Besides given that this is the actual loans we want to predict, only the structure is going to be explored, when we have the final full dataset that is when we are going to do the train/test split and start the proper EDA

In [ ]:
application = pd.read_csv(defecto / "application_train.csv")

In [ ]:
application.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Let's see how many observations do we have, which is crucial on the type of split we are going to make

In [ ]:
print(f"Applications table has {application.shape[0]} rows and {application.shape[1]} columns")

Applications table has 307511 rows and 122 columns


Okey we have a lot of data actually, so we are going to have a train/val/test approach given the vast amount we have.
Also this implies we can use more variables given the 300k rows and not end overfitting the models so easily.

Given there are 122 columns, we are not going to put a dictionary here, you can find the whole descriptions here :
https://www.kaggle.com/c/home-credit-default-risk/data

On this first dive only basic structure modifications are going to be made, for example one-hot encoding on binary variables, we will leave the rest when we have the full dataset joined

### 1.2.1.1 Checking Nulls

First, let's check for null values

In [ ]:
pd.set_option('display.max_rows', None)
resume = application.isnull().sum()
resume[resume > 0].sort_values(ascending=False)

COMMONAREA_MEDI                 214865
COMMONAREA_MODE                 214865
COMMONAREA_AVG                  214865
NONLIVINGAPARTMENTS_MODE        213514
NONLIVINGAPARTMENTS_MEDI        213514
NONLIVINGAPARTMENTS_AVG         213514
FONDKAPREMONT_MODE              210295
LIVINGAPARTMENTS_AVG            210199
LIVINGAPARTMENTS_MEDI           210199
LIVINGAPARTMENTS_MODE           210199
FLOORSMIN_MEDI                  208642
FLOORSMIN_MODE                  208642
FLOORSMIN_AVG                   208642
YEARS_BUILD_MODE                204488
YEARS_BUILD_MEDI                204488
YEARS_BUILD_AVG                 204488
OWN_CAR_AGE                     202929
LANDAREA_AVG                    182590
LANDAREA_MEDI                   182590
LANDAREA_MODE                   182590
BASEMENTAREA_MODE               179943
BASEMENTAREA_MEDI               179943
BASEMENTAREA_AVG                179943
EXT_SOURCE_1                    173378
NONLIVINGAREA_MEDI              169682
NONLIVINGAREA_AVG        

We see actually there is a lot of NA, we are going to drop a lot since the vast majority of them are nulls, but we are going to keep that that the null is another way of telling us another type of information

In [ ]:
pd.set_option('display.max_rows', 20)

COMMONAREA and NONLIVINGAPARMENTS are going to be directly dropped, since we would have to know how the data was harvested to see if its that simply was not proportionated by the person solicting the loan, or it means something about the home of that person, since we don't know and its mostly null we are going to directly drop it.

FONDKAPREMONT could betdireclty dropped too because its exclusive to a certain geographical place, but its going to be maintained since that null implies that it does not apply to that person, giving that us more information, we are going to leave it as it is, because of the different values, we cant just conclude they are all equal and can convert into FONDKAREMONT applies or not

In [ ]:
to_drop = ["COMMONAREA_MEDI","COMMONAREA_MODE","COMMONAREA_AVG","NONLIVINGAPARTMENTS_MODE","NONLIVINGAPARTMENTS_MEDI","NONLIVINGAPARTMENTS_AVG"]
application.drop(to_drop,axis=1,inplace=True)

LIVINGAPARTMENTS nulls could also imply the person is homeless, or data was not provided, or lives somewhere without with no rooms.
So we are going to nust drop it because it does not gives us certain information

In [ ]:
application.drop(["LIVINGAPARTMENTS_AVG","LIVINGAPARTMENTS_MEDI","LIVINGAPARTMENTS_MODE"],axis=1,inplace=True)

FLOORSMIN is going to be dropped too because the same reasoning as LIVINGAPARTMENTS, it does not give us certain info about why is that a null

In [ ]:
application.drop(["FLOORSMIN_AVG","FLOORSMIN_MEDI","FLOORSMIN_MODE"],axis=1,inplace=True)

YEARSBUILD too because the same reasoning as before

In [ ]:
application.drop(["YEARS_BUILD_AVG","YEARS_BUILD_MEDI","YEARS_BUILD_MODE"],axis=1,inplace=True)

Now with OWN_CAR_AGE, nulls could imply they dont have a car, given the vast majority of nulls, and that that info is already conveyed on the variable FLAG_OWN_CAR, we could just use this second variable, since the first one is unusable due to the quantity of nulls. But first let's check if the hypothesis made its true

In [ ]:
application[application["FLAG_OWN_CAR"] == "N"]["OWN_CAR_AGE"].isnull().all()

np.True_

The hypothesis was true, the null means they do not have a car, so we are going to preventively code the NaN as zero

In [ ]:
application["OWN_CAR_AGE"] = application["OWN_CAR_AGE"].fillna(value=0)

Now with the LANDAREA variables the same reasoning as the variables we dropped before, we are going to drop them

In [ ]:
application.drop(["LANDAREA_AVG","LANDAREA_MEDI","LANDAREA_MODE"],axis=1,inplace=True)

We have remaining

In [ ]:
resume = application.isnull().sum()
resume[resume >= application.shape[0]*0.5].sort_values(ascending=False)

FONDKAPREMONT_MODE    210295
BASEMENTAREA_AVG      179943
BASEMENTAREA_MODE     179943
BASEMENTAREA_MEDI     179943
EXT_SOURCE_1          173378
                       ...  
ENTRANCES_MODE        154828
LIVINGAREA_AVG        154350
LIVINGAREA_MODE       154350
LIVINGAREA_MEDI       154350
HOUSETYPE_MODE        154297
Length: 22, dtype: int64

Dropped by the same reasoning as before

In [ ]:
application.drop(["BASEMENTAREA_AVG","BASEMENTAREA_MEDI","BASEMENTAREA_MODE","NONLIVINGAREA_AVG","NONLIVINGAREA_MODE","NONLIVINGAREA_MEDI","BASEMENTAREA_AVG","BASEMENTAREA_MODE","BASEMENTAREA_MEDI","ELEVATORS_AVG","ELEVATORS_MODE","ELEVATORS_MEDI","WALLSMATERIAL_MODE","APARTMENTS_MODE","APARTMENTS_AVG","APARTMENTS_MEDI","ENTRANCES_MEDI","ENTRANCES_MODE","ENTRANCES_AVG","LIVINGAREA_MODE","LIVINGAREA_MEDI","LIVINGAREA_AVG","HOUSETYPE_MODE","FLOORSMAX_MODE","FLOORSMAX_MEDI","FLOORSMAX_AVG","YEARS_BEGINEXPLUATATION_MODE","YEARS_BEGINEXPLUATATION_AVG","YEARS_BEGINEXPLUATATION_MEDI","TOTALAREA_MODE","EMERGENCYSTATE_MODE"],axis=1,inplace=True)

The AMT_REQ_CREDIT_BUREAU ones means how many times the historial of the client has been revised, a null could imply it was not checked, that is the client did not solicit a loan on the approximate time span of the variable, so we it suits a 0 in this case to be imputed

In [ ]:
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]
application[bureau_cols] = application[bureau_cols].fillna(value=0)

The EXT_SOURCE ones, since they are scores given by external sources, later we are going to see which one we can impute by median or mean, or which have too many nulls to be used

We no have only remaining

In [ ]:
resume = application.isnull().sum()
resume[resume > 0].sort_values(ascending=False)

FONDKAPREMONT_MODE          210295
EXT_SOURCE_1                173378
OCCUPATION_TYPE              96391
EXT_SOURCE_3                 60965
NAME_TYPE_SUITE               1292
OBS_30_CNT_SOCIAL_CIRCLE      1021
DEF_60_CNT_SOCIAL_CIRCLE      1021
OBS_60_CNT_SOCIAL_CIRCLE      1021
DEF_30_CNT_SOCIAL_CIRCLE      1021
EXT_SOURCE_2                   660
AMT_GOODS_PRICE                278
AMT_ANNUITY                     12
CNT_FAM_MEMBERS                  2
DAYS_LAST_PHONE_CHANGE           1
dtype: int64

OCCUPATION_TYPE is going to be filled with Unspecified until the proper EDA can be done, because i suspect the occupation plays a big role in someone paying back or not

In [ ]:
application["OCCUPATION_TYPE"] = application["OCCUPATION_TYPE"].fillna(value="Unspecified")

Now with the OBS and DEF one we are going to use the hypothesis that simply there weren't any people observed, or that they did not have contact with anyone that had x time delay in payment, on both ways we can impute with zeros

In [ ]:
application["OBS_30_CNT_SOCIAL_CIRCLE"] = application["OBS_30_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["OBS_60_CNT_SOCIAL_CIRCLE"] = application["OBS_60_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["DEF_30_CNT_SOCIAL_CIRCLE"] = application["DEF_30_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["DEF_60_CNT_SOCIAL_CIRCLE"] = application["DEF_60_CNT_SOCIAL_CIRCLE"].fillna(value=0)

For the rest we can just simply use the median or mean when we do the proper eda when we join every table and have the final dataset

Finally on the first table we finally have :

In [ ]:
print(application.shape[1],"columns")

76 columns


Let's check if there are any duplicated ids

In [ ]:
application["SK_ID_CURR"].nunique() == int(len(application["SK_ID_CURR"]))

True

Since there aren't any duplicated we must format the data on one single id, probably having to make aggregate features

### 1.2.2 installments_payments.csv

Let's start with "installments_payments.csv" on which we can explore the historial of other loans and its respective payment, lets change the type of data so we dont consume all the memory

In [ ]:
dtypes = {
    'SK_ID_PREV': 'int32',
    'SK_ID_CURR': 'int32',
    'NUM_INSTALMENT_VERSION': 'uint16',
    'NUM_INSTALMENT_NUMBER': 'uint16',
    'DAYS_INSTALMENT': 'float32',
    'DAYS_ENTRY_PAYMENT': 'float32',
    'AMT_INSTALMENT': 'float32',
    'AMT_PAYMENT': 'float32'
}

payments = pd.read_csv(defecto / "installments_payments.csv",dtype=dtypes)

Before diving into it, let's first explore its columns and what do they mean

In [ ]:
print("The columns are :",end=" ")
for column in list(payments.columns):
    print(column,end=", ")

The columns are : SK_ID_PREV, SK_ID_CURR, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT, AMT_INSTALMENT, AMT_PAYMENT, 

In [ ]:
payments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1,6,-1180.0,-1187.0,6948.359863,6948.359863
1,1330831,151639,0,34,-2156.0,-2156.0,1716.525024,1716.525024
2,2085231,193053,2,1,-63.0,-63.0,25425.000000,25425.000000
3,2452527,199697,1,3,-2418.0,-2426.0,24350.130859,24350.130859
4,2714724,167756,1,2,-1383.0,-1366.0,2165.040039,2160.584961


Here a little dictionary to understand the meaning of each variable

| Column | Description |
|--------|-------------|
| `SK_ID_CURR` | ID of the current loan (links to main application table) |
| `SK_ID_PREV` | ID of the previous credit in Home Credit |
| `NUM_INSTALMENT_VERSION` | Version of the installment calendar (0 = credit card) |
| `NUM_INSTALMENT_NUMBER` | Installment number (which payment in the sequence) |
| `DAYS_INSTALMENT` | Day the installment was supposed to be paid (relative to application date) |
| `DAYS_ENTRY_PAYMENT` | Day the installment was actually paid (relative to application date) |
| `AMT_INSTALMENT` | Amount that was supposed to be paid for that installment |
| `AMT_PAYMENT` | Amount actually paid for that installment |

First lets explore if there is going to be any type of temporal leakage.

DAYS_INSTALMENT could be used to determine if there is an active loan ongoing besides the actual requested.
But it could happen that DAYS_INSTALMENT > 0, that is that the date to repay the loan is after the solicitude, but DAYS_ENTRY_PAYMENT < 0, which means it was fully repayed before the loan, so its not active anymore

In [ ]:
payments.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
NUM_INSTALMENT_VERSION       0
NUM_INSTALMENT_NUMBER        0
DAYS_INSTALMENT              0
DAYS_ENTRY_PAYMENT        2905
AMT_INSTALMENT               0
AMT_PAYMENT               2905
dtype: int64

First lets check if we have to group by ID or there is one row per id

In [ ]:
print(f"On Payments table we have {payments.shape[0]} rows and {payments.shape[1]} columns")

On Payments table we have 13605401 rows and 8 columns


Seeing the number of rows it is almost sure there are duplicated ids

In [ ]:
payments["SK_ID_CURR"].nunique()

339587

There is duplicated, definitely, so we would have to group by the current loan id, to make sure we can join it with the main table, and get some aggregate metrics, some options are :
- Ratio of times client payed less than it should have, for this it is needed variable that keeps track of difference between the established money and the money payed
- Ratio of payments made late
- Ratio of payments not made, for this we need an impayment variable
- Quantity of loans made by the actual client, for this we only need to count different previous loans id
- Total quantity unpaid

In [ ]:
payments["DELAY_TIME"] = (payments["DAYS_ENTRY_PAYMENT"] - payments["DAYS_INSTALMENT"])
payments["DELAY_TIME"]

0           -7.0
1            0.0
2            0.0
3           -8.0
4           17.0
            ... 
13605396     NaN
13605397     NaN
13605398     NaN
13605399     NaN
13605400     NaN
Name: DELAY_TIME, Length: 13605401, dtype: float32

On this new variable, negative values indicate the payment has been made early and positive values otherwise.

Given thtat DAYS_INSTALMENT has no nulls, they come from DAYS_ENTRY_PAYMENT which shows us they have not been payed to this day, so we can make a new variable indicating that the current client has another loan active

Let's consider two distinct cases, one when DAYS_INSTALMENT is < 0, so the time to pay expired and DAYS_ENTRY_PAYMENT is null, it counts as and impayed charge.

On other hand, we have when DAYS_INSTALMENT is > 0, and DAYS_ENTRY_PAYMENT is null, so that indicates he has still time to pay it to this day, this is the client has another active loan

And it could be done too a percent of lately paid charges

In [ ]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 9 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int32  
 1   SK_ID_CURR              int32  
 2   NUM_INSTALMENT_VERSION  uint16 
 3   NUM_INSTALMENT_NUMBER   uint16 
 4   DAYS_INSTALMENT         float32
 5   DAYS_ENTRY_PAYMENT      float32
 6   AMT_INSTALMENT          float32
 7   AMT_PAYMENT             float32
 8   DELAY_TIME              float32
dtypes: float32(5), int32(2), uint16(2)
memory usage: 415.2 MB


We are going to process it on chunks so the memory does not explode

In [ ]:
unpaid_array = np.zeros(len(payments), dtype='uint8')
chunk_size = 2_000_000
for start in range(0, len(payments), chunk_size):
    end = start + chunk_size

    days_inst = payments['DAYS_INSTALMENT'].iloc[start:end].values
    days_entry = payments['DAYS_ENTRY_PAYMENT'].iloc[start:end].values
    
    unpaid_array[start:end] = ((days_inst <= 0) & np.isnan(days_entry)).astype('uint8')
payments['UNPAID_INSTALMENT'] = unpaid_array
payments['UNPAID_INSTALMENT']

0           0
1           0
2           0
3           0
4           0
           ..
13605396    1
13605397    1
13605398    1
13605399    1
13605400    1
Name: UNPAID_INSTALMENT, Length: 13605401, dtype: uint8

In [ ]:
amt_payment_clean = payments['AMT_PAYMENT'].fillna(0)
payments["PAYMENT_DIFF"] = (payments["AMT_INSTALMENT"] - amt_payment_clean).astype("Float32")
payments["PAYMENT_DIFF"]

0                    0.0
1                    0.0
2                    0.0
3                    0.0
4               4.455078
                ...     
13605396            67.5
13605397            67.5
13605398    43737.433594
13605399            67.5
13605400        11504.25
Name: PAYMENT_DIFF, Length: 13605401, dtype: Float32

In [ ]:
payments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,DELAY_TIME,UNPAID_INSTALMENT,PAYMENT_DIFF
0,1054186,161674,1,6,-1180.0,-1187.0,6948.359863,6948.359863,-7.0,0,0.0
1,1330831,151639,0,34,-2156.0,-2156.0,1716.525024,1716.525024,0.0,0,0.0
2,2085231,193053,2,1,-63.0,-63.0,25425.000000,25425.000000,0.0,0,0.0
3,2452527,199697,1,3,-2418.0,-2426.0,24350.130859,24350.130859,-8.0,0,0.0
4,2714724,167756,1,2,-1383.0,-1366.0,2165.040039,2160.584961,17.0,0,4.455078
...,...,...,...,...,...,...,...,...,...,...,...
13605396,2186857,428057,0,66,-1624.0,NaN,67.500000,NaN,NaN,1,67.5
13605397,1310347,414406,0,47,-1539.0,NaN,67.500000,NaN,NaN,1,67.5
13605398,1308766,402199,0,43,-7.0,NaN,43737.433594,NaN,NaN,1,43737.433594
13605399,1062206,409297,0,43,-1986.0,NaN,67.500000,NaN,NaN,1,67.5


In [ ]:
loan_level = payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg({
    'NUM_INSTALMENT_NUMBER': 'max',      # Plazo total del préstamo (ej. 12, 24, 36 cuotas)
    'AMT_INSTALMENT': 'mean',           # Cuota mensual media de este préstamo
    'PAYMENT_DELAY': 'max',             # Peor retraso de pago en este préstamo
    'PAYMENT_DIFF': 'sum',              # Deuda total no pagada en este préstamo
    'UNPAID_INSTALMENT': 'sum'          # Número de cuotas impagadas en este préstamo
}).reset_index()
# Renombrar columnas a nivel de préstamo
loan_level.columns = [
    'SK_ID_CURR', 'SK_ID_PREV', 
    'LOAN_TERM', 'LOAN_AVG_PAYMENT', 
    'LOAN_MAX_DELAY', 'LOAN_TOTAL_DEBT', 'LOAN_UNPAID_COUNT'
]